In [ ]:
%load_ext watermark


In [ ]:
import itertools as it
import os

import matplotlib as mpl
from matplotlib import pyplot as plt
import pandas as pd
from phyloframe import _auxlib as pfa
from phyloframe import legacy as pfl
from pyfonts import load_google_font
import seaborn as sns
from teeplot import teeplot as tp

import pylib


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2026-03-12-btr-fossil-clade")
teeplot_subdir


In [ ]:
pfa.seed_random(1)


In [ ]:
font = load_google_font("Merriweather", weight=300)
mpl.font_manager.fontManager.addfont(font.get_file())
plt.rcParams["font.family"] = font.get_name()


## Prep Data


In [ ]:
df_pure = pfl.alifestd_join_roots(
    pd.read_parquet("https://osf.io/vjhgs/download"),
)
df_pure


In [ ]:
df_pure["x"] = df_pure["position"] // df_pure["nCol"]
df_pure["x_"] = df_pure["x"] / df_pure["nRow"]
df_pure["y"] = df_pure["position"] % df_pure["nCol"]
df_pure["y_"] = df_pure["y"] / df_pure["nCol"]

df_pure["origin_time"] = df_pure["dstream_rank"]


## Plot Layer Tree


In [ ]:
for regime, seed in it.product(
    ("pure",),
    (5,),
):
    df = df_pure.copy()
    df_ = df.copy()
    df1 = pfl.alifestd_downsample_tips_clade_asexual(
        df,
        n_downsample=3_000,
        seed=seed,
    )

    df2 = pfl.alifestd_downsample_tips_clade_asexual(
        df,
        n_downsample=200,
        seed=seed + 8,
    )

    df3 = pfl.alifestd_downsample_tips_clade_asexual(
        df,
        n_downsample=1_000,
        seed=seed + 5,
    )

    df_ = pfl.alifestd_downsample_tips_asexual(
        df_,
        n_downsample=3_000,
        seed=seed,
    )
    df_["x__"] = df_["x_"].copy()
    df_["y__"] = df_["y_"].copy()

    mask = (
        df_["id"].isin(df1["id"])
        | df_["id"].isin(df2["id"])
        | df_["id"].isin(df3["id"])
    )
    df_.loc[~mask, "x_"] = pd.NA
    df_.loc[~mask, "y_"] = pd.NA


    df_ = pfl.alifestd_collapse_unifurcations(df_)
    df1 = pfl.alifestd_collapse_unifurcations(df1)
    df2 = pfl.alifestd_collapse_unifurcations(df2)
    df3 = pfl.alifestd_collapse_unifurcations(df3)

    for layout in "vertical",:

        with tp.teed(
            pylib.chloropleth.draw_ctree,
            df_,
            x="x__",
            y="y__",
            cmap=lambda *args, **kwargs: sns.set_hls_values(
                pylib.cmap.bcyr.get_color(*args, **kwargs),
                l=0.85,
                s=0.2,
            ),
            layout=layout,
            scatter_kws=dict(
                # alpha=0.3,
                edgecolor="none",
                s=4,
            ),
            scatter_shuffle=1,
            tree_kws=dict(
                edge=dict(
                    color="gainsboro",
                    linewidth=0.5,
                ),
                margins=-0.05,
            ),
            teeplot_outattrs={"regime": regime, "seed": seed},
            teeplot_subdir=teeplot_subdir,
        ) as teed:
            pylib.chloropleth.draw_ctree(
                df_,
                x="x_",
                y="y_",
                cmap=pylib.cmap.bcyr.get_color,
                layout=layout,
                scatter_kws=dict(
                    edgecolor="none",
                    s=4,
                ),
                scatter_shuffle=1,
                tree_kws=dict(
                    edge=dict(
                        alpha=0.0,
                        color="gainsboro",
                        linewidth=0.5,
                    ),
                    margins=-0.05,
                ),
            )
            teed.invert_yaxis()
            teed.figure.set_size_inches(3, 1.33)

    with tp.teed(
        pylib.chloropleth.draw_cscatter,
        pd.concat(
            [df1, df2, df3],
        ).dropna(subset=["x_", "y_"]),
        x="x",
        y="y",
        cmap=pylib.cmap.bcyr.get_color,
        despine=True,
        major=100,
        minor=25,
        xmax=1170,
        ymax=755,
        scatter_kws=dict(
            s=20,
        ),
        teeplot_outattrs={"cmap": "bcyr", "regime": regime, "seed": seed},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        teed.set_aspect("equal")
        fig = teed.figure
        fig.set_size_inches(3, 2)
        fig.tight_layout()
